# PREDICTA: TRACE — posterior inference

Predict a hidden parameter `mu` from 100 noisy, gappy, irregularly-sampled observations of a
2-D nonlinear oscillator. Metric: **MAE**. 15,000 train / 4,000 test.

This notebook **supersedes** the earlier exploratory one. It does not re-derive anything that
was already settled there; those conclusions are stated as constants below, with the number
that was measured. What is left is the part that pays.

---

## What was removed, and what it scored

| removed | what it scored | why it is gone |
|---|---|---|
| `beta` recovery by variance matching; `xdot = y` by integration; the period/amplitude probe table | `beta ~ 0.19`, `corr = 0.83`, periods 5.11 → 6.54 | `xdot = y` is hardcoded in §B; `beta` is now **fitted by likelihood** in §C0, which variance matching never settled |
| weak-form SINDy **as the prediction** | failed — recovers a `mu` coefficient of 0.43 instead of 1.00 (regression dilution) | kept only as a feature, where it is free |
| CPU profile-MLE over `(mu, x0, y0)` | MAE **0.1358**, `corr 0.971`, **2.7 hours** | replaced by the GPU posterior in §C — strictly more informative and ~30x faster |
| simulation pretraining, 120k trajectories at `sigma = 0.80` | zero-shot 0.2452, fine-tuned 0.1385 — never beat the GBM | the posterior now does the physics job directly; the simulator survives only as the §C3 diagnostic |
| GAF images + ImageNet ResNet-18 | fold-0 0.1417 pretrained / 0.1319 scratch, 370 s/fold | no better than the 1-D CNN, at more cost |
| MOMENT-1 embeddings | never produced a score | fixed 512 context and reversible instance norm, which erases the amplitude signal |
| median baseline | 0.6277 | printed as a constant in §F |

## What is new

1. **`sigma` is not 0.80 everywhere.** It is 0.800 on train and **0.843 on test** (§B). Every
   previous model was trained to invert the wrong noise level.
2. **A posterior, not a point estimate** (§C). The generative model is fully known and the noise
   is Gaussian with known `sigma`, so `p(mu | data)` is computable. Under MAE the optimal
   predictor is its **median** — and the whole curve becomes a feature.
3. **The Bayes floor is measured** (§C3), so we know how much room is actually left.
4. **`beta` is fitted by maximum likelihood** at known `mu` (§C0) instead of inherited from
   variance matching, which left it ambiguous between 0.19 and ~0.3.
5. The CNN predicts the **residual** against the posterior median instead of `mu` from scratch.
6. Every stage checkpoints and rewrites `submission.csv`, so a session that dies part-way still
   leaves the best result reached so far on disk.

---
## Environment and data

Runs unchanged on Kaggle or locally. On Kaggle the competition files land read-only under
`/kaggle/input/<competition>/` and anything written must go to `/kaggle/working/`; locally they
sit in `./data`. The cell below finds whichever exists rather than hardcoding either.

The OpenMP guard below is set before anything imports. LightGBM and torch each ship their own
OpenMP runtime, and on macOS having both loaded segfaults the first torch CPU op; pinning OpenMP
to one thread avoids it (verified: `KMP_DUPLICATE_LIB_OK` alone does **not**). It is gated to
darwin, so Kaggle's Linux images keep full threading.

In [ ]:
import os, sys, glob

if sys.platform == 'darwin':
    # LightGBM + torch both load libomp; on macOS that segfaults the first torch CPU op.
    # Must be set BEFORE either import. Linux/Kaggle is unaffected and keeps full threading.
    os.environ.setdefault('OMP_NUM_THREADS', '1')

KAGGLE = os.path.isdir('/kaggle/input')
NEEDED = ('train.csv', 'test.csv', 'train_labels.csv', 'sample_submission.csv')


def find_data():
    """Locate the directory holding the competition CSVs."""
    cands = []
    if KAGGLE:
        for root, _, files in os.walk('/kaggle/input'):
            if 'train.csv' in files and 'test.csv' in files:
                cands.append(root)
    for p in ('data', '.', '../data', '../input'):
        if os.path.isfile(os.path.join(p, 'train.csv')):
            cands.append(os.path.abspath(p))
    if not cands:
        raise FileNotFoundError(
            'no directory with train.csv found. On Kaggle, attach the competition dataset '
            '(Add Input); locally, put the CSVs in ./data')
    # prefer a directory that has every file we need
    full = [c for c in cands if all(os.path.isfile(os.path.join(c, f)) for f in NEEDED)]
    return (full or cands)[0]


if KAGGLE:                                    # the usual Kaggle inventory
    print('/kaggle/input contents:')
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for fn in sorted(filenames)[:20]:
            p = os.path.join(dirname, fn)
            print(f'  {p}  ({os.path.getsize(p) / 1e6:.1f} MB)')

DATA_DIR = find_data()
OUT_DIR = '/kaggle/working' if KAGGLE else '.'
os.makedirs(OUT_DIR, exist_ok=True)

print(f'\nDATA_DIR = {DATA_DIR}')
print(f'OUT_DIR  = {OUT_DIR}')
missing = [f for f in NEEDED if not os.path.isfile(os.path.join(DATA_DIR, f))]
print('files:', {f: f'{os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6:.1f} MB'
                 for f in NEEDED if f not in missing})
if missing:
    print(f'WARNING: not found in DATA_DIR: {missing}')

In [ ]:
# LightGBM must be imported BEFORE torch: both ship their own OpenMP runtime, and if torch's
# loads first, LightGBM's training call segfaults the kernel on macOS. Harmless elsewhere.
import lightgbm as lgb
import torch, torch.nn as nn

import os, time, math
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.signal import savgol_filter
from scipy.stats import kurtosis, skew, norm
from scipy.linalg import lstsq
from sklearn.model_selection import KFold

%matplotlib inline

DATA = DATA_DIR                                        # resolved in the cell above
GRID = np.linspace(0.0, 5.0, 100)                      # common regular time grid
TRAPZ = getattr(np, 'trapezoid', None) or np.trapz

DEV = ('cuda' if torch.cuda.is_available() else
       'mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available()
       else 'cpu')

# ---------------------------------------------------------------------------------------
PRESET = 'max'             # 'smoke' | 'quick' | 'full' | 'max'
# ---------------------------------------------------------------------------------------

# Rough wall clock on one modern data-centre GPU. The CNN dominates everything else;
# the posterior stage is launch-bound, so its cost barely moves with the number of rows.
#   smoke  ~4 min      subsampled, exercises every cell — run this first on a new machine
#   quick  ~35 min     one CNN seed, coarse posterior
#   full   ~2.5 h      the safe choice if the session might be cut short
#   max    ~7 h        finest posterior, 5 CNN seeds; needs a full-length GPU session
PRESETS = {
    'smoke': dict(n_mu=21,  it_first=6,  it_warm=3, ode_h=0.02,   floor_n=400,   beta_n=5,
                  cnn_width=32,  cnn_epochs=3,  cnn_folds=1,  cnn_seeds=1, gbm_rounds=150,
                  subsample=1500),
    'quick': dict(n_mu=61,  it_first=10, it_warm=4, ode_h=0.01,   floor_n=2000,  beta_n=9,
                  cnn_width=64,  cnn_epochs=30, cnn_folds=10, cnn_seeds=1, gbm_rounds=4000,
                  subsample=0),
    'full':  dict(n_mu=161, it_first=14, it_warm=6, ode_h=0.005,  floor_n=8000,  beta_n=13,
                  cnn_width=96,  cnn_epochs=60, cnn_folds=10, cnn_seeds=3, gbm_rounds=8000,
                  subsample=0),
    'max':   dict(n_mu=321, it_first=18, it_warm=8, ode_h=0.0025, floor_n=20000, beta_n=21,
                  cnn_width=128, cnn_epochs=90, cnn_folds=10, cnn_seeds=5, gbm_rounds=15000,
                  subsample=0),
}
CFG = dict(PRESETS[PRESET])
CFG.update(batch=256, lr=3e-3, weight_decay=1e-4, ode_keep=2, ode_chunk=120_000,
           hess_step=0.05, ic_box=1.5, ic_clamp=3.0)

NFOLD, SPLIT_SEED, NOISE_SEED = 10, 0, 7
BETA = 0.19                   # the earlier variance-matching estimate; refined by likelihood in C0
FIT_BETA = True               # sweep beta against the known-mu likelihood instead of trusting 0.19
MATCH_TEST_NOISE = True       # see section B: train is quieter than test, so lift train to match
SUBMISSION = os.path.join(OUT_DIR, 'submission.csv' if PRESET != 'smoke'
                          else 'submission_smoke.csv')

# --- chart tokens: one sequential blue ramp for magnitude, blue/orange for identity ---
SURFACE, INK, INK2, MUTED = '#ffffff', '#0b0b0b', '#52514e', '#898781'
GRIDC, AXISC = '#e1e0d9', '#c3c2b7'
BLUE, ORANGE, GREEN, PURPLE = '#2a78d6', '#eb6834', '#1baf7a', '#4a3aa7'
SEQ = ['#cde2fb','#b7d3f6','#9ec5f4','#86b6ef','#6da7ec','#5598e7','#3987e5',
       '#2a78d6','#256abf','#1c5cab','#184f95','#104281','#0d366b']
CMAP = LinearSegmentedColormap.from_list('seq_blue', SEQ)
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'figure.dpi': 110, 'font.size': 9.5,
    'font.family': 'sans-serif', 'font.sans-serif': ['Helvetica Neue', 'Helvetica', 'Arial', 'DejaVu Sans'],
    'axes.edgecolor': AXISC, 'axes.linewidth': 0.8, 'axes.labelcolor': INK2,
    'axes.titlecolor': INK, 'axes.titlesize': 10.5, 'axes.titlelocation': 'left',
    'axes.titlepad': 8, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRIDC, 'grid.linewidth': 0.8, 'grid.linestyle': '-',
    'xtick.color': MUTED, 'ytick.color': MUTED, 'text.color': INK,
    'xtick.labelcolor': MUTED, 'ytick.labelcolor': MUTED,
    'lines.linewidth': 2.0, 'legend.frameon': False, 'legend.fontsize': 9, 'axes.axisbelow': True,
})
pd.set_option('display.width', 130)

print(f'preset {PRESET!r} on {DEV}   torch {torch.__version__}   lightgbm {lgb.__version__}')
print({k: CFG[k] for k in PRESETS[PRESET]})

---
## Block A — Load

Every trajectory has exactly 100 rows already in time order, so the CSV reshapes straight to
`(N, 100)`; grouping is unnecessary and much slower. The asserts are what make that safe.

Two representations are kept and both are used later:

* **raw** `(t, x, y)` — the irregular samples with `NaN` gaps. The likelihood in §C uses these,
  at their true times. Nothing is interpolated before it reaches the physics.
* **gridded** `(Xg, Yg, Mg)` — linearly interpolated onto `GRID`, plus a local observation
  density. Only the feature block and the CNN use these.

In [ ]:
def load(split):
    """CSV -> (ids, t, x, y) as dense (N, 100) arrays."""
    df = pd.read_csv(f'{DATA}/{split}.csv')
    ids = df.trajectory_id.values[::100].copy()
    assert (df.groupby('trajectory_id').size() == 100).all(), 'not all trajectories have 100 rows'
    t = df.time.values.reshape(-1, 100)
    x = df.x.values.reshape(-1, 100)
    y = df.y.values.reshape(-1, 100)
    assert (np.diff(t, axis=1) > 0).all(), 'rows are not time-ordered inside a trajectory'
    assert (df.trajectory_id.values.reshape(-1, 100) == ids[:, None]).all(), 'rows not blocked by id'
    assert (np.isnan(x) ^ np.isnan(y)).sum() == 0, 'x and y do not drop together'
    return ids, t, x, y


def to_grid(t, x, y):
    """Linear-interpolate each trajectory onto GRID; also return local observation density."""
    N = len(t)
    Xg = np.empty((N, 100)); Yg = np.empty((N, 100)); Mg = np.empty((N, 100))
    for i in range(N):
        m = ~np.isnan(x[i])
        Xg[i] = np.interp(GRID, t[i][m], x[i][m])
        Yg[i] = np.interp(GRID, t[i][m], y[i][m])
        Mg[i] = np.interp(GRID, t[i], m.astype(float))     # 1 = dense here, 0 = inside a gap
    return Xg, Yg, Mg


t0 = time.time()
A = {}
for split in ['train', 'test']:
    ids, t, x, y = load(split)
    A[split] = dict(ids=ids, t=t, x=x, y=y)
    print(f'{split:5s} {len(ids):6d} trajectories   raw {x.shape}   missing {np.isnan(x).mean():.2%}')

lab = pd.read_csv(f'{DATA}/train_labels.csv').set_index('trajectory_id').target
mu = lab.loc[A['train']['ids']].values                     # aligned to array row order
A['train']['mu'] = mu

if CFG['subsample']:                                       # smoke preset only
    n = CFG['subsample']
    for s in ['train', 'test']:
        k = min(n, len(A[s]['ids']))
        for key in ['ids', 't', 'x', 'y']:
            A[s][key] = A[s][key][:k]
    mu = mu[:len(A['train']['ids'])]; A['train']['mu'] = mu
    print(f'SUBSAMPLED to {len(A["train"]["ids"])} train / {len(A["test"]["ids"])} test')

ids_test = A['test']['ids']
print(f'labels aligned: mu[:3] = {np.round(mu[:3], 4)}   ({time.time() - t0:.1f}s)')

---
## Block B — what is already known, and the one thing that was wrong

Established in the earlier notebook, and taken as given here:

```
xdot = y
ydot = mu*(1 - x^2)*y - x - beta*x^3          # Van der Pol + Duffing stiffening
```

| quantity | value | how it was established |
|---|---|---|
| `xdot = y` | confirmed | `corr(cumtrapz(y,t), x) = 0.83`, slope 0.99 — by **integration**, never differentiation |
| `beta` | `0.19` | forward simulation matched to the per-timestep variance curves of both channels |
| `mu` prior | `U(0.50, 3.00)` | label histogram |
| initial conditions | `x0, y0 ~ U(-1.5, 1.5)` | per-step variance at `k = 0` minus `sigma^2` → 0.735 / 0.723 vs 0.750 |
| sampling | `dt ~ U(0.0416, 0.060)`, window renormalised to exactly `[0, 5]` | `diff(time)` |
| gaps | ~7%, `x` and `y` drop together, isolated drops plus runs of 2-3 | run-length histogram |

### The correction

`sigma ~ 0.80` was measured **on train only** and then used everywhere — including as the
hardcoded `SIG = 0.80` of the simulator. Measured separately per split with a 4th-difference
filter restricted to windows of five *consecutive observed* indices (so gaps cannot inflate it
through the larger effective spacing), the two splits do not agree. The cell below re-measures
it, because this is the one number the rest of the notebook is calibrated to.

The estimator is unbiased here: on train it reads 0.797-0.805 across all five quintiles of the
*true* `mu`, so there is no signal leaking into it, and the whole per-trajectory distribution
shifts by the same factor rather than a subpopulation moving — it is a scale change, not a mixture.

In [ ]:
K4 = np.array([1, -4, 6, -4, 1]) / np.sqrt(70)      # annihilates cubics, unit gain on white noise


def sigma_gapfree(Araw):
    """Robust noise sigma using only windows of 5 CONSECUTIVE OBSERVED indices.

    Restricting to gap-free windows matters: across a gap the effective spacing is 2-4x larger
    and the filter's signal leakage grows as dt^4, which would masquerade as extra noise."""
    obs = ~np.isnan(Araw)
    ok = obs[:, 0:96] & obs[:, 1:97] & obs[:, 2:98] & obs[:, 3:99] & obs[:, 4:100]
    B = np.nan_to_num(Araw)
    r = sum(K4[j] * B[:, j:96 + j] for j in range(5))
    return float(np.median(np.abs(r[ok])) / 0.6745)     # MAD -> sigma, robust to the tails


SIGMA = {}
rows = []
for split in ['train', 'test']:
    sx = sigma_gapfree(A[split]['x']); sy = sigma_gapfree(A[split]['y'])
    SIGMA[split] = 0.5 * (sx + sy)
    d = A[split]
    miss = np.isnan(d['x']).mean(); dt = np.diff(d['t'], axis=1)
    rows.append(dict(split=split, n=len(d['ids']), sigma_x=sx, sigma_y=sy,
                     sigma=SIGMA[split], missing=miss, dt_mean=dt.mean(),
                     dt_lo=dt.min(), dt_hi=dt.max()))
tab = pd.DataFrame(rows)
display(tab.style.hide(axis='index').format({
    'sigma_x': '{:.4f}', 'sigma_y': '{:.4f}', 'sigma': '{:.4f}',
    'missing': '{:.2%}', 'dt_mean': '{:.5f}', 'dt_lo': '{:.4f}', 'dt_hi': '{:.4f}'}))

ratio = SIGMA['test'] / SIGMA['train']
print(f'test / train noise ratio {ratio:.4f}   ->   test carries {ratio ** 2 - 1:+.1%} more noise VARIANCE')

# leakage check: if the filter were picking up signal, this would trend with mu
qs = np.quantile(mu, np.linspace(0, 1, 6))
print('\nsigma_hat on train, stratified by TRUE mu (flat => no signal leakage):')
for a, b in zip(qs[:-1], qs[1:]):
    m = (mu >= a) & (mu < b + (b == qs[-1]))
    print(f'  mu in [{a:.2f}, {b:.2f})  n={m.sum():5d}   sigma = {sigma_gapfree(A["train"]["x"][m]):.4f}')

SIG_MODEL = SIGMA['test']       # everything downstream is calibrated to the split we are scored on
print(f'\nmodelling sigma = {SIG_MODEL:.4f} (the test value) for both splits')

### B2 — lift train onto the test noise level

Two ways to handle the mismatch, and only one of them is right for the leaderboard.

*Leave it.* Models learn to invert `sigma = 0.800` data and are then handed `sigma = 0.843`
data. They **under-shrink**: the optimal pull toward the prior grows with the noise, so every
prediction is a little too confident, worst at the edges of the `mu` range where the truncated
prior does the most work. Cross-validation, measured entirely at 0.800, never shows it.

*Match it.* Add `N(0, sqrt(sigma_test^2 - sigma_train^2))` to the **raw** train samples — before
any interpolation, so the added noise is independent exactly the way the real noise is. Train and
test become identically distributed, every CV number becomes an honest estimate of the
leaderboard, and the posterior in §C is calibrated with a single `sigma`.

It costs a little real signal on train. That is the correct trade: the score is computed on test.

In [ ]:
EXTRA = math.sqrt(max(SIGMA['test'] ** 2 - SIGMA['train'] ** 2, 0.0))
print(f'extra noise needed on train: sqrt({SIGMA["test"]:.4f}^2 - {SIGMA["train"]:.4f}^2) = {EXTRA:.4f}')

if MATCH_TEST_NOISE and EXTRA > 0:
    rng = np.random.default_rng(NOISE_SEED)
    d = A['train']
    keep = ~np.isnan(d['x'])                       # keep the gap pattern exactly as it was
    d['x'] = d['x'] + np.where(keep, rng.normal(0, EXTRA, d['x'].shape), 0.0)
    d['y'] = d['y'] + np.where(keep, rng.normal(0, EXTRA, d['y'].shape), 0.0)
    print(f'train re-measured after lifting: sigma_x {sigma_gapfree(d["x"]):.4f}  '
          f'sigma_y {sigma_gapfree(d["y"]):.4f}   (target {SIGMA["test"]:.4f})')
else:
    print('SKIPPED -- CV below will be optimistic relative to the leaderboard.')

for split in ['train', 'test']:                    # grid representation, after the lift
    d = A[split]
    d['Xg'], d['Yg'], d['Mg'] = to_grid(d['t'], d['x'], d['y'])
print(f'gridded: train {A["train"]["Xg"].shape}, test {A["test"]["Xg"].shape}')

---
## Block C — the posterior over `mu`

The generative model is known, the noise is Gaussian and independent, and `sigma` is now known
per split. So for one trajectory the likelihood is exactly

```
log p(data | mu, x0, y0) = -1/(2 sigma^2) * SUM_over_observed [ (x_k - X(t_k))^2 + (y_k - Y(t_k))^2 ] + const
```

where `X, Y` solve the ODE from `(x0, y0)`. Nothing is approximated except the ODE solve.

Four things this does that a point MLE did not:

1. **It scores at the real observation times.** `t_k` is given in the data. The old fit compared
   against `Xg`, the interpolated grid — which correlates neighbouring noise, throws away the
   exact times, and adds an interpolation bias on top.
2. **It keeps the whole curve.** Sweeping `mu` and optimising `(x0, y0)` at each value is what the
   old MLE already did; it then discarded everything but the `argmin`. The curve's width and
   asymmetry are the uncertainty, and that is exactly what a blender needs.
3. **It marginalises the initial conditions instead of profiling them.** Profiling biases toward
   the `mu` where the fit is tightest. The Laplace volume term corrects for it, and it is not a
   small correction here: at large `mu` the trajectory collapses onto the limit cycle quickly, so
   it is *less* sensitive to where it started, so the `(x0, y0)` volume consistent with the data is
   *larger*. Dropping that term systematically penalises large `mu`.
4. **It uses the prior.** `mu ~ U(0.5, 3.0)` and `x0, y0 ~ U(-1.5, 1.5)` are both known. Under MAE
   the optimal point prediction is the **posterior median**, and the median of a truncated
   posterior shrinks at the edges by exactly the right amount — which is where the old models'
   largest bias was (+0.107 at `mu < 0.75`, -0.110 at `mu > 2.75`).

The whole thing is one batched ODE solve per candidate, so it belongs on the GPU. On CPU the
equivalent sweep cost 2.7 hours; here it is minutes, which is what makes the fine grid affordable.

In [ ]:
TT = torch.float32


def integrate(mu_t, x0_t, y0_t, beta, h, keep, T=5.0):
    """Batched RK4. Returns states on a coarse output grid of spacing h*keep, shape (B, nk)."""
    n = int(round(T / h)); nk = n // keep + 1
    x, y = x0_t.clone(), y0_t.clone()
    Xs = torch.empty((len(mu_t), nk), device=x.device, dtype=TT)
    Ys = torch.empty_like(Xs)
    Xs[:, 0] = x; Ys[:, 0] = y
    j = 1
    for i in range(n):
        k1x = y;                             k1y = mu_t * (1 - x * x) * y - x - beta * x ** 3
        xa = x + .5 * h * k1x; ya = y + .5 * h * k1y
        k2x = ya;                            k2y = mu_t * (1 - xa * xa) * ya - xa - beta * xa ** 3
        xb = x + .5 * h * k2x; yb = y + .5 * h * k2y
        k3x = yb;                            k3y = mu_t * (1 - xb * xb) * yb - xb - beta * xb ** 3
        xc = x + h * k3x;      yc = y + h * k3y
        k4x = yc;                            k4y = mu_t * (1 - xc * xc) * yc - xc - beta * xc ** 3
        x = x + h / 6 * (k1x + 2 * k2x + 2 * k3x + k4x)
        y = y + h / 6 * (k1y + 2 * k2y + 2 * k3y + k4y)
        if (i + 1) % keep == 0:
            Xs[:, j] = x; Ys[:, j] = y; j += 1
    return Xs, Ys


def sample_at(Xs, Ys, Tq, dtk):
    """Linear interpolation of the solved paths onto each trajectory's own observation times.

    Output spacing is h*keep = 0.02 by default; the interpolation error is ~(dtk^2/8)*|x''|,
    about 1e-3 at the worst point of the range, against a measurement sigma of 0.84."""
    pos = (Tq / dtk).clamp_(0, Xs.shape[1] - 1.0001)
    i0 = pos.floor().long()
    frac = pos - i0.to(TT)
    xa = torch.gather(Xs, 1, i0); xb = torch.gather(Xs, 1, i0 + 1)
    ya = torch.gather(Ys, 1, i0); yb = torch.gather(Ys, 1, i0 + 1)
    return xa + (xb - xa) * frac, ya + (yb - ya) * frac


@torch.no_grad()
def sse(mu_t, x0_t, y0_t, D, beta, h, keep, chunk):
    """Sum of squared residuals over the OBSERVED samples only, both channels. Chunked."""
    out = torch.empty(len(mu_t), device=mu_t.device, dtype=TT)
    for i in range(0, len(mu_t), chunk):
        sl = slice(i, min(i + chunk, len(mu_t)))
        Xs, Ys = integrate(mu_t[sl], x0_t[sl], y0_t[sl], beta, h, keep)
        xq, yq = sample_at(Xs, Ys, D['Tq'][sl], h * keep)
        out[sl] = (((xq - D['Xo'][sl]) ** 2 + (yq - D['Yo'][sl]) ** 2) * D['M'][sl]).sum(1)
    return out


def pack(d, reps):
    """Move one split to the device, pre-tiled `reps` times so probe batches need no copy."""
    keep = (~np.isnan(d['x'])).astype(np.float32)
    T_ = torch.tensor(np.tile(d['t'], (reps, 1)), device=DEV, dtype=TT)
    X_ = torch.tensor(np.tile(np.nan_to_num(d['x']), (reps, 1)), device=DEV, dtype=TT)
    Y_ = torch.tensor(np.tile(np.nan_to_num(d['y']), (reps, 1)), device=DEV, dtype=TT)
    M_ = torch.tensor(np.tile(keep, (reps, 1)), device=DEV, dtype=TT)
    return dict(Tq=T_, Xo=X_, Yo=Y_, M=M_)

### C1 — the sweep

For each `mu` on the grid, `(x0, y0)` is optimised by pattern search: four probes (`+-dx`, `+-dy`)
evaluated **in one batched solve**, take the best, halve the step wherever nothing improved. The
search is warm-started from the previous `mu`, so after the first grid point the optimum has
barely moved and a handful of rounds suffices.

Then, at each optimum, an 8-point stencil gives the `(x0, y0)` Hessian of the SSE and the Laplace
marginal follows in closed form:

```
log L(mu)  =  -SSE_min(mu) / (2 sigma^2)  -  0.5 * log det H(mu)  +  log P(box)
```

`log P(box)` is the mass of the Laplace Gaussian that lands inside the known `[-1.5, 1.5]^2`
prior — it demotes a `mu` whose best explanation needs an initial condition the generator could
never have drawn.

In [ ]:
@torch.no_grad()
def posterior(d, sigma, beta=None, n_mu=None, it_first=None, it_warm=None, h=None,
              trunc=True, tag='', verbose=True):
    """Profile + Laplace posterior over mu for every trajectory in `d`.

    Returns (mu_grid, logL (N, G), SSEmin (N, G), ICs (N, G, 2))."""
    beta = BETA if beta is None else beta
    n_mu = n_mu or CFG['n_mu']; it_first = it_first or CFG['it_first']
    it_warm = it_warm or CFG['it_warm']; h = h or CFG['ode_h']
    keep, chunk, hstep = CFG['ode_keep'], CFG['ode_chunk'], CFG['hess_step']
    N = len(d['t']); K = 8                       # 8 = widest probe batch (the Hessian stencil)
    D = pack(d, K)
    dev = DEV

    def _sse(cmu, cx, cy, r):                    # r = how many tiles of N this probe uses
        sub = {k: v[:r * N] for k, v in D.items()}
        return sse(cmu, cx.clamp(-CFG['ic_clamp'], CFG['ic_clamp']),
                   cy.clamp(-CFG['ic_clamp'], CFG['ic_clamp']), sub, beta, h, keep, chunk)

    xs0 = savgol_filter(d['Xg'], 15, 3, axis=1)[:, 0]     # smoothed head: a start, not the answer
    ys0 = savgol_filter(d['Yg'], 15, 3, axis=1)[:, 0]
    cx = torch.tensor(xs0, device=dev, dtype=TT)
    cy = torch.tensor(ys0, device=dev, dtype=TT)

    mu_grid = np.linspace(0.5, 3.0, n_mu)
    logL = np.empty((N, n_mu), np.float64)
    SSEm = np.empty((N, n_mu), np.float32)
    ICs = np.empty((N, n_mu, 2), np.float32)
    inv2s2 = 1.0 / (2 * sigma ** 2)
    t0 = time.time()

    for j, m_ in enumerate(mu_grid):
        mv = torch.full((N,), float(m_), device=dev, dtype=TT)
        step = torch.full((N,), 0.5 if j == 0 else 0.12, device=dev, dtype=TT)
        cur = _sse(mv, cx, cy, 1)
        mv4 = mv.repeat(4)
        for _ in range(it_first if j == 0 else it_warm):
            CX = torch.stack([cx + step, cx - step, cx, cx])          # (4, N)
            CY = torch.stack([cy, cy, cy + step, cy - step])
            S = _sse(mv4, CX.reshape(-1), CY.reshape(-1), 4).view(4, N)
            v, a = S.min(0)
            imp = v < cur
            cx = torch.where(imp, CX.gather(0, a[None])[0], cx)
            cy = torch.where(imp, CY.gather(0, a[None])[0], cy)
            cur = torch.where(imp, v, cur)
            step = torch.where(imp, step, step * 0.5)

        # --- Laplace: 8-point stencil for the (x0, y0) Hessian of the SSE at the optimum ---
        e = hstep
        dx = torch.tensor([e, -e, 0, 0, e, e, -e, -e], device=dev, dtype=TT)[:, None]
        dy = torch.tensor([0, 0, e, -e, e, -e, e, -e], device=dev, dtype=TT)[:, None]
        S8 = _sse(mv.repeat(K), (cx[None] + dx).reshape(-1),
                  (cy[None] + dy).reshape(-1), K).view(K, N)
        Hxx = (S8[0] - 2 * cur + S8[1]) / e ** 2
        Hyy = (S8[2] - 2 * cur + S8[3]) / e ** 2
        Hxy = (S8[4] - S8[5] - S8[6] + S8[7]) / (4 * e ** 2)
        det = (Hxx * Hyy - Hxy ** 2).clamp_min(1e-8)

        ll = -cur * inv2s2 - 0.5 * torch.log(det)      # float32: MPS has no float64
        if trunc:                                  # prior mass of the Laplace blob inside the box
            vx = (2 * sigma ** 2 * Hyy / det).clamp_min(1e-8).sqrt()
            vy = (2 * sigma ** 2 * Hxx / det).clamp_min(1e-8).sqrt()
            b = CFG['ic_box']
            cdf = lambda z: 0.5 * (1 + torch.erf(z / math.sqrt(2)))
            px = (cdf((b - cx) / vx) - cdf((-b - cx) / vx)).clamp_min(1e-6)
            py = (cdf((b - cy) / vy) - cdf((-b - cy) / vy)).clamp_min(1e-6)
            ll = ll + torch.log(px * py)

        logL[:, j] = ll.cpu().numpy().astype(np.float64)
        SSEm[:, j] = cur.cpu().numpy()
        ICs[:, j, 0] = cx.cpu().numpy(); ICs[:, j, 1] = cy.cpu().numpy()
        if verbose and (j % max(1, n_mu // 6) == 0 or j == n_mu - 1):
            print(f'  {tag} mu {j + 1:3d}/{n_mu} = {m_:.3f}   '
                  f'best SSE {cur.mean().item():.1f}   '
                  f'({time.time() - t0:.0f}s)', flush=True)
    if verbose:
        print(f'  {tag} done in {time.time() - t0:.0f}s', flush=True)
    return mu_grid, logL, SSEm, ICs

### C2 — turning the curve into numbers

The posterior median is the prediction. Everything else on the curve becomes a feature: its
width, its skew, how peaked it is, how well the best fit actually explained the data, and the
local shape of the density sampled *relative to the median*, which makes it comparable across
trajectories.

In [ ]:
def _quantile(cdf, grid, lev):
    """Row-wise inverse CDF by linear interpolation. cdf: (N, G) increasing to 1."""
    G = cdf.shape[1]
    idx = np.clip((cdf < lev).sum(1), 1, G - 1)
    r = np.arange(len(cdf))
    c0, c1 = cdf[r, idx - 1], cdf[r, idx]
    g0, g1 = grid[idx - 1], grid[idx]
    f = np.where(c1 > c0, (lev - c0) / np.maximum(c1 - c0, 1e-12), 0.0)
    return g0 + np.clip(f, 0, 1) * (g1 - g0)


OFFS = np.array([-0.60, -0.40, -0.28, -0.18, -0.10, -0.04, 0.0,
                 0.04, 0.10, 0.18, 0.28, 0.40, 0.60])


def summarize(mu_grid, logL, SSEm, ICs, n_obs, sigma):
    """(N, G) log-likelihood curve -> point prediction + a feature block."""
    w = np.gradient(mu_grid)[None, :]                      # uniform grid -> constant, cancels
    p = np.exp(logL - logL.max(1, keepdims=True)) * w
    p /= p.sum(1, keepdims=True)
    cdf = np.clip(np.cumsum(p, 1), 0, 1); cdf /= cdf[:, -1:]

    med = _quantile(cdf, mu_grid, 0.50)
    q = {L: _quantile(cdf, mu_grid, L) for L in (0.10, 0.25, 0.75, 0.90)}
    mean = (p * mu_grid).sum(1)
    sd = np.sqrt(np.maximum((p * (mu_grid - mean[:, None]) ** 2).sum(1), 0))
    mapv = mu_grid[logL.argmax(1)]
    ent = -(p * np.log(p + 1e-300)).sum(1)
    jbest = logL.argmax(1)
    r = np.arange(len(logL))
    sse_at_map = SSEm[r, jbest]
    ic = ICs[r, jbest]                                      # ICs fitted at the MAP
    # chi-square per observed value: 1.0 means the physics explains the data down to the noise.
    # 2 * n_obs * sigma^2 is the expected SSE of a perfect fit (two channels, n_obs points each).
    chi2 = sse_at_map / np.maximum(2 * n_obs * sigma ** 2, 1e-9)

    dens = np.stack([np.interp(med[i] + OFFS, mu_grid, p[i] / w[0]) for i in range(len(p))])
    dens = dens / (dens.max(1, keepdims=True) + 1e-12)

    F = np.column_stack([
        med, mean, mapv, sd, ent,
        q[0.10], q[0.25], q[0.75], q[0.90],
        q[0.75] - q[0.25], q[0.90] - q[0.10],
        mean - med, med - mapv,                             # skew of the posterior
        np.log1p(sse_at_map), chi2, np.log(np.maximum(sd, 1e-6)),
        ic[:, 0], ic[:, 1], np.hypot(ic[:, 0], ic[:, 1]),
        med - 0.5, 3.0 - med,                               # distance to each prior edge
        dens,
    ]).astype(np.float32)
    names = (['po_med', 'po_mean', 'po_map', 'po_sd', 'po_entropy',
              'po_q10', 'po_q25', 'po_q75', 'po_q90', 'po_iqr', 'po_w80',
              'po_mean_minus_med', 'po_med_minus_map',
              'po_log_sse', 'po_chi2', 'po_log_sd',
              'po_x0', 'po_y0', 'po_r0', 'po_edge_lo', 'po_edge_hi']
             + [f'po_dens{k}' for k in range(len(OFFS))])
    assert F.shape[1] == len(names)
    return med, F, np.array(names), p

### C0 — stop guessing `beta`

`beta = 0.19` came from matching simulated variance curves, and the earlier notebook flagged that
the same matching also tolerated values near 0.3. It was never settled because settling it by
simulation is expensive. With the likelihood on the GPU it is nearly free, and there is a far
more direct route than variance matching:

**train has 15,000 trajectories whose `mu` is known.** So fix `mu` at its true value, optimise only
`(x0, y0)`, and compare the pooled best SSE across candidate `beta`. The minimum is the maximum-
likelihood `beta` for the whole dataset. No labels are *spent* — `mu` is given, and `beta` is one
global scalar, so there is nothing to overfit and no need to hold anything out.

If the fitted value lands near 0.19 that confirms the original estimate and the `mu` posterior is
built on solid ground. If it lands elsewhere, every number in the earlier notebook was computed
with a slightly wrong forward model.

In [ ]:
@torch.no_grad()
def fit_beta(d, mu_known, sigma, betas, iters=None, h=None, tag=''):
    """Pooled maximum-likelihood beta at KNOWN mu: optimise only (x0, y0) per candidate."""
    iters = iters or CFG['it_first']; h = h or CFG['ode_h']
    keep, chunk = CFG['ode_keep'], CFG['ode_chunk']
    N = len(d['t']); D = pack(d, 4)
    mv = torch.tensor(np.asarray(mu_known, np.float32), device=DEV, dtype=TT)
    mv4 = mv.repeat(4)
    xs0 = savgol_filter(d['Xg'], 15, 3, axis=1)[:, 0]
    ys0 = savgol_filter(d['Yg'], 15, 3, axis=1)[:, 0]
    out = []
    t0 = time.time()
    for b in betas:
        cx = torch.tensor(xs0, device=DEV, dtype=TT)
        cy = torch.tensor(ys0, device=DEV, dtype=TT)
        step = torch.full((N,), 0.5, device=DEV, dtype=TT)

        def f(cm, a, c, r):
            return sse(cm, a.clamp(-CFG['ic_clamp'], CFG['ic_clamp']),
                       c.clamp(-CFG['ic_clamp'], CFG['ic_clamp']),
                       {k: v[:r * N] for k, v in D.items()}, float(b), h, keep, chunk)

        cur = f(mv, cx, cy, 1)
        for _ in range(iters):
            CX = torch.stack([cx + step, cx - step, cx, cx])
            CY = torch.stack([cy, cy, cy + step, cy - step])
            S = f(mv4, CX.reshape(-1), CY.reshape(-1), 4).view(4, N)
            v, a = S.min(0)
            imp = v < cur
            cx = torch.where(imp, CX.gather(0, a[None])[0], cx)
            cy = torch.where(imp, CY.gather(0, a[None])[0], cy)
            cur = torch.where(imp, v, cur)
            step = torch.where(imp, step, step * 0.5)
        out.append(float(cur.mean()))
        print(f'  {tag} {b:.4f}   mean SSE {out[-1]:.5f}   ({time.time() - t0:.0f}s)', flush=True)
    return np.array(out)


if FIT_BETA:
    nb_ = min(6000, len(mu))                       # one global scalar; 6k rows is ample
    dsub = {k: A['train'][k][:nb_] for k in ('t', 'x', 'y', 'Xg', 'Yg', 'Mg')}
    betas = np.linspace(0.05, 0.45, CFG['beta_n'])
    curve = fit_beta(dsub, mu[:nb_], SIG_MODEL, betas, tag='beta')

    j = int(curve.argmin())
    if 0 < j < len(betas) - 1:                     # parabolic refinement through the 3 best points
        y0_, y1_, y2_ = curve[j - 1], curve[j], curve[j + 1]
        denom = (y0_ - 2 * y1_ + y2_)
        shift = 0.5 * (y0_ - y2_) / denom if abs(denom) > 1e-12 else 0.0
        BETA_FIT = float(betas[j] + np.clip(shift, -1, 1) * (betas[1] - betas[0]))
    else:
        BETA_FIT = float(betas[j])
    print(f'\nmaximum-likelihood beta = {BETA_FIT:.4f}   (grid argmin {betas[j]:.4f}, '
          f'prior estimate 0.19)')

    n_pts = 2 * (~np.isnan(dsub['x'])).sum(1).mean()
    print(f'SSE per observed value at the optimum: {curve[j] / n_pts:.4f}   '
          f'(2*sigma^2 = {2 * SIG_MODEL ** 2:.4f} would be the noise floor for ungridded data)')

    fig, ax = plt.subplots(figsize=(5.2, 3.2))
    ax.plot(betas, curve / n_pts, color=BLUE, marker='o', ms=4, mfc=SURFACE, mew=1.4)
    ax.axvline(BETA_FIT, color=ORANGE, lw=1.4, ls='--')
    ax.axvline(0.19, color=MUTED, lw=1.2, ls=':')
    ax.set(xlabel='beta', ylabel='SSE per observed value',
           title=f'likelihood in beta at known mu -> {BETA_FIT:.3f}')
    plt.tight_layout(); plt.show()

    BETA = BETA_FIT
print(f'using BETA = {BETA:.4f} for the posterior and the simulator')

### C3 — the two checks worth paying for

**Is the ODE step size fine enough?** `h = 0.01` against a reference solve at `h = 0.0025`, on a
subsample. If the posterior median moves by much less than the eventual MAE, the discretisation
is free.

**How much room is left at all?** Simulate trajectories from the recovered generator at the
*test* noise level, where `mu` is known exactly, and run the identical posterior machinery on
them. Because the simulator and the inference model are then the same model, the resulting MAE is
the **Bayes floor**: the best any method could do if the recovered physics is exactly right.

One caveat that matters, and it is self-diagnosing. This is only a floor once the posterior is
*converged* — a coarse `mu` grid, a loose `(x0, y0)` search or a large ODE step all inflate it,
because they degrade the estimator rather than the information. The tell is unmistakable: if a
trained model scores **below** the measured floor, the floor is not a floor, it is just the
accuracy of an under-resolved posterior, and `n_mu` / `it_warm` / `ode_h` need tightening before
the number means anything. At the `smoke` settings this is expected; at `full` or `max` it is a
red flag.

In [ ]:
@torch.no_grad()
def simulate(n, seed, sigma, miss_rate, h=0.0025, keep=2):
    """Draw n trajectories from the recovered generator in exactly the observed format."""
    g = np.random.default_rng(seed)
    mu_s = g.uniform(0.5, 3.0, n).astype(np.float32)
    x0 = g.uniform(-1.5, 1.5, n).astype(np.float32)
    y0 = g.uniform(-1.5, 1.5, n).astype(np.float32)
    dt = g.uniform(0.0416, 0.060, (n, 99))
    t = np.concatenate([np.zeros((n, 1)), np.cumsum(dt, 1)], 1)
    t = (t / t[:, -1:] * 5.0).astype(np.float32)           # window renormalised to exactly [0,5]

    Xs, Ys = integrate(*[torch.tensor(v, device=DEV, dtype=TT) for v in (mu_s, x0, y0)],
                       BETA, h, keep)
    xq, yq = sample_at(Xs, Ys, torch.tensor(t, device=DEV, dtype=TT), h * keep)
    x = xq.cpu().numpy().astype(np.float64); y = yq.cpu().numpy().astype(np.float64)
    x += g.normal(0, sigma, x.shape); y += g.normal(0, sigma, y.shape)

    p_iso = miss_rate * 0.52                               # tuned to the observed run-length mix:
    p_run = miss_rate * 0.17                               # ~63% singletons, mean run ~1.74
    bad = g.random((n, 100)) < p_iso
    st = g.random((n, 100)) < p_run
    for k in (0, 1, 2):
        bad[:, k:] |= (st[:, :100 - k] if k else st)
    x[bad] = np.nan; y[bad] = np.nan
    d = dict(t=t.astype(np.float64), x=x, y=y, mu=mu_s)
    d['Xg'], d['Yg'], d['Mg'] = to_grid(d['t'], d['x'], d['y'])
    return d


t0 = time.time()
SIM = simulate(CFG['floor_n'], 2024, SIG_MODEL, float(np.isnan(A['test']['x']).mean()))
print(f'simulated {len(SIM["mu"])}: missing {np.isnan(SIM["x"]).mean():.2%} '
      f'(test {np.isnan(A["test"]["x"]).mean():.2%}), '
      f'sigma_hat {sigma_gapfree(SIM["x"]):.4f} (target {SIG_MODEL:.4f})   {time.time() - t0:.0f}s')

# --- step-size check, on the simulated set where the truth is known ---
nchk = min(600, len(SIM['mu']))
chk = {k: v[:nchk] for k, v in SIM.items()}
gA, lA, sA, iA = posterior(chk, SIG_MODEL, n_mu=41, h=CFG['ode_h'], tag='h=coarse', verbose=False)
gB, lB, sB, iB = posterior(chk, SIG_MODEL, n_mu=41, h=0.0025, tag='h=fine', verbose=False)
mA = summarize(gA, lA, sA, iA, (~np.isnan(chk['x'])).sum(1), SIG_MODEL)[0]
mB = summarize(gB, lB, sB, iB, (~np.isnan(chk['x'])).sum(1), SIG_MODEL)[0]
print(f'\nODE step check: h={CFG["ode_h"]} vs h=0.0025 -> median moves by '
      f'{np.abs(mA - mB).mean():.5f} (max {np.abs(mA - mB).max():.4f})')
print(f'   MAE at h={CFG["ode_h"]}: {np.abs(mA - chk["mu"]).mean():.4f}   '
      f'at h=0.0025: {np.abs(mB - chk["mu"]).mean():.4f}')

In [ ]:
gS, lS, sS, iS = posterior(SIM, SIG_MODEL, tag='sim')
med_sim, F_sim, PO_NAMES, p_sim = summarize(gS, lS, sS, iS, (~np.isnan(SIM['x'])).sum(1), SIG_MODEL)

BAYES_FLOOR = float(np.abs(med_sim - SIM['mu']).mean())
print(f'\n*** BAYES FLOOR (posterior median on simulated data, sigma = {SIG_MODEL:.3f}): '
      f'MAE {BAYES_FLOOR:.4f} ***')
print(f'    RMSE {np.sqrt(((med_sim - SIM["mu"]) ** 2).mean()):.4f}   '
      f'correlation {np.corrcoef(med_sim, SIM["mu"])[0, 1]:.4f}')
print(f'    for comparison: MAP instead of median would score '
      f'{np.abs(F_sim[:, PO_NAMES.tolist().index("po_map")] - SIM["mu"]).mean():.4f}, '
      f'mean {np.abs(F_sim[:, 1] - SIM["mu"]).mean():.4f}')
print(f'    median chi^2 per observed value: {np.median(F_sim[:, PO_NAMES.tolist().index("po_chi2")]):.3f}'
      '  (1.0 = fit is at the noise floor)')

fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.6))
ax = axes[0]
ax.hexbin(SIM['mu'], med_sim, gridsize=40, cmap=CMAP, mincnt=1, linewidths=0)
ax.plot([0.5, 3], [0.5, 3], color=ORANGE, lw=1.4, ls='--')
ax.set(xlabel='true mu', ylabel='posterior median', title=f'simulated: MAE {BAYES_FLOOR:.4f}')
ax = axes[1]
ed = np.linspace(0.5, 3.0, 11); cen = 0.5 * (ed[1:] + ed[:-1])
err = [np.abs(med_sim[(SIM['mu'] >= a) & (SIM['mu'] < b)]
              - SIM['mu'][(SIM['mu'] >= a) & (SIM['mu'] < b)]).mean() for a, b in zip(ed[:-1], ed[1:])]
ax.plot(cen, err, color=BLUE, marker='o', ms=4, mfc=SURFACE, mew=1.4)
ax.set(xlabel='true mu', ylabel='MAE', title='where the information actually is', ylim=(0, None))
ax = axes[2]
for i in np.linspace(0, len(p_sim) - 1, 9).astype(int):
    ax.plot(gS, p_sim[i] / p_sim[i].max(), color=CMAP((SIM['mu'][i] - 0.5) / 2.5), lw=1.4)
ax.set(xlabel='mu', ylabel='posterior (scaled)', title='nine posteriors, coloured by true mu')
plt.tight_layout(); plt.show()

### C4 — run it on the real data

The same call, on train and test. The posterior median is recorded as a model in its own right:
it uses **no labels at all**, only the recovered physics and the measured noise level.

In [ ]:
RESULTS = {}
folds = list(KFold(NFOLD, shuffle=True, random_state=SPLIT_SEED).split(np.arange(len(mu))))


def write_submission(pred, path, quiet=False):
    """Write a submission in sample_submission's row order, clipped to the known support."""
    s = pd.DataFrame({'trajectory_id': ids_test, 'target': np.clip(pred, 0.5, 3.0)})
    order = pd.read_csv(f'{DATA}/sample_submission.csv').trajectory_id
    if len(order) == len(s):
        s = s.set_index('trajectory_id').loc[order].reset_index()
    assert s.target.notna().all() and s.target.between(0.5, 3.0).all()
    s.to_csv(path, index=False)
    if not quiet:
        print(f'  wrote {path}  ({len(s)} rows)')
    return s


def ckpt(name, **arrays):
    """Persist a stage so a cut-short session does not lose everything before it."""
    p = os.path.join(OUT_DIR, f'ckpt_{name}.npz')
    np.savez_compressed(p, **arrays)
    print(f'  checkpoint -> {p}')


def record(name, oof, test=None, mask=None, note=''):
    """Score a model on the rows it actually covered and keep it for the blend."""
    m = np.ones(len(mu), bool) if mask is None else mask
    mae = float(np.abs(oof[m] - mu[m]).mean())
    rmse = float(np.sqrt(((oof[m] - mu[m]) ** 2).mean()))
    RESULTS[name] = dict(oof=oof, test=test, mask=m, mae=mae, rmse=rmse, note=note)
    cov = '' if m.all() else f'  [{m.sum()} of {len(m)} rows]'
    print(f'{name:24s} MAE {mae:.4f}   RMSE {rmse:.4f}{cov}  {note}')
    return mae


t0 = time.time()
PO = {}
for split in ['train', 'test']:
    d = A[split]
    g_, l_, s_, i_ = posterior(d, SIG_MODEL, tag=split)
    med_, F_, PO_NAMES, p_ = summarize(g_, l_, s_, i_, (~np.isnan(d['x'])).sum(1), SIG_MODEL)
    PO[split] = dict(grid=g_, med=med_, F=F_, p=p_)
    print(f'{split}: median chi^2 {np.median(F_[:, PO_NAMES.tolist().index("po_chi2")]):.3f}', flush=True)
print(f'posterior on both splits in {time.time() - t0:.0f}s')

Ptr, Pte = PO['train']['F'], PO['test']['F']
record('posterior median', PO['train']['med'], PO['test']['med'],
       note='physics only, no labels used')
ckpt('posterior', oof=PO['train']['med'], test=PO['test']['med'],
     Ftr=Ptr, Fte=Pte, names=PO_NAMES, beta=np.array([BETA]), sigma=np.array([SIG_MODEL]))
write_submission(PO['test']['med'], SUBMISSION)
print('  ^ a valid submission now exists from the physics alone, before any training')
print(f"gap to the Bayes floor: {RESULTS['posterior median']['mae'] - BAYES_FLOOR:+.4f}"
      f"  ({100 * (RESULTS['posterior median']['mae'] / BAYES_FLOOR - 1):+.1f}%)")

---
## Block E — engineered features

Unchanged from the previous notebook and still worth their cost: 218 features aimed at the three
real signal sources. The earlier gain ranking confirmed the physics — `ys_kurt` (relaxation
character of the waveform), `ys_absmax` (peak `|y|`, which unlike `|x|` does move with `mu`),
`r_std`, then the autocorrelation period and the transient block statistics. Note what stayed
absent: anything built on the amplitude of `x`. The limit cycle sits at `|x| ~ 2.0` for every
`mu`, so `x` amplitude is close to information-free.

Weak-form SINDy stays, as a feature only. As a *predictor* it fails outright — the regressors are
themselves built from noisy `x, y`, and regression dilution attenuates the fit to a `mu`
coefficient of 0.43 against a true 1.00 — but it is monotone in `mu` and it is nearly free here.

In [ ]:
S2 = SIG_MODEL ** 2                  # noise variance, now the test-matched value
FREQS = np.linspace(0.4, 3.0, 40)    # angular frequency; the limit cycle sits at 2pi/6.5..2pi/5.1
NB = 10                              # time blocks
AC_LAGS = [2, 4, 6, 8, 10, 12, 15, 18, 21, 25, 30, 35, 40, 50]


def lomb(tt, v, freqs):
    """Lomb-Scargle power at given angular frequencies. Correct under irregular sampling,
    where a plain FFT is not."""
    v = v - v.mean(); P = np.empty(len(freqs))
    for k, w in enumerate(freqs):
        wt = w * tt
        s2w, c2w = np.sin(2 * wt).sum(), np.cos(2 * wt).sum()
        tau = 0.5 * np.arctan2(s2w, c2w) / w
        ct, st = np.cos(w * (tt - tau)), np.sin(w * (tt - tau))
        P[k] = 0.5 * ((v * ct).sum() ** 2 / max((ct ** 2).sum(), 1e-9)
                      + (v * st).sum() ** 2 / max((st ** 2).sum(), 1e-9))
    return P


def sindy(tt, xx, yy):
    """Weak-form (integral) SINDy fit of (mu, beta).

    Given xdot = y the system is linear in (mu, beta), so multiplying ydot by a compact bump phi
    and integrating by parts removes the derivative entirely:
        -int(y phi') + int(x phi) = mu * int((1-x^2)y phi) - beta * int(x^3 phi).
    Noise debiasing is analytic: E[x^3] = x^3 + 3 x sigma^2, E[x^2 y] = x^2 y + sigma^2 y."""
    n = len(tt); half, p, nwin = 11, 5, 10
    Amat, bvec = [], []
    for c in np.linspace(half, n - 1 - half, nwin).astype(int):
        sl = slice(c - half, c + half + 1)
        ts, xs_, ys_ = tt[sl], xx[sl], yy[sl]
        s = (ts - ts[0]) / (ts[-1] - ts[0]) * 2 - 1
        phi = (1 - s ** 2) ** p
        dphi = p * (1 - s ** 2) ** (p - 1) * (-2 * s) * (2 / (ts[-1] - ts[0]))
        x2y = xs_ ** 2 * ys_ - S2 * ys_                 # debiased
        x3 = xs_ ** 3 - 3 * S2 * xs_                    # debiased
        I = lambda f: TRAPZ(f, ts)
        Amat.append([I((ys_ - x2y) * phi), -I(x3 * phi)])
        bvec.append(-I(ys_ * dphi) + I(xs_ * phi))
    Amat, bvec = np.array(Amat), np.array(bvec)
    try:
        sol = lstsq(Amat, bvec)[0]
        res = bvec - Amat @ sol
        return [sol[0], sol[1], np.sqrt(np.mean(res ** 2))]
    except Exception:
        return [0.0, 0.0, 0.0]

In [ ]:
def one(tr, xr, yr, xg, yg):
    """218 features for a single trajectory."""
    f = []
    m = ~np.isnan(xr); to, xo, yo = tr[m], xr[m], yr[m]
    f.append(m.mean())

    xs = savgol_filter(xg, 15, 3); ys = savgol_filter(yg, 15, 3)
    xs2 = savgol_filter(xg, 25, 3)                      # second smoothing scale

    for v in (xg, yg, xs, ys):                          # global moments
        f += [v.std(), np.abs(v).mean(), np.percentile(np.abs(v), 90), np.abs(v).max(),
              kurtosis(v), skew(v), (v ** 2).mean()]
    f += [max(xg.var() - S2, 0), max(yg.var() - S2, 0), np.corrcoef(xg, yg)[0, 1],
          np.corrcoef(xs, ys)[0, 1], (xg * yg).mean()]

    for v in (xs, ys, xs ** 2 + ys ** 2):               # convergence onto the limit cycle
        bl = v.reshape(NB, -1)
        f += list(bl.std(1)) + list(np.abs(bl).mean(1))
        f += [bl.std(1)[-1] - bl.std(1)[0], np.polyfit(np.arange(NB), bl.std(1), 1)[0]]
    e = np.abs(xs).reshape(NB, -1).max(1)               # envelope growth
    f += list(e) + [e[-1] / (e[0] + 1e-6), e[-1] - e[0]]

    for v in (xs, xs2, ys):                             # zero crossings / period
        zc = np.where(np.diff(np.sign(v)) != 0)[0]
        f.append(len(zc))
        f += [GRID[zc[0]], GRID[zc[-1]]] if len(zc) >= 1 else [5.0, 5.0]
        f.append(np.mean(np.diff(GRID[zc])) * 2 if len(zc) >= 2 else 0.0)
    f += [GRID[np.argmax(xs)], GRID[np.argmin(xs)], xs.max(), xs.min(),
          GRID[np.argmax(np.abs(ys))], np.abs(ys).max()]

    for v in (xo, yo):                                  # Lomb-Scargle on the RAW samples
        P = lomb(to, v, FREQS); P = P / (P.sum() + 1e-9)
        f += [FREQS[np.argmax(P)], P.max(), (FREQS * P).sum(),
              np.sqrt(((FREQS - (FREQS * P).sum()) ** 2 * P).sum()),
              -(P * np.log(P + 1e-12)).sum()]
        f += list(P[::3])

    for v in (xs, ys):                                  # autocorrelation: period and damping
        vv = v - v.mean()
        ac = np.correlate(vv, vv, 'full')[len(vv) - 1:]
        ac = ac / (ac[0] + 1e-9)
        f += list(ac[AC_LAGS])
        neg = np.where(ac < 0)[0]
        f.append(GRID[neg[0]] if len(neg) else 5.0)
        f.append(np.argmin(ac) * (GRID[1] - GRID[0]))

    rr = np.sqrt(xs ** 2 + ys ** 2)                     # phase plane and waveform shape
    f += [rr.mean(), rr.std(), rr.max(), rr[-12:].mean() - rr[:12].mean()]
    th = np.unwrap(np.arctan2(ys, xs))
    f += [-(th[-1] - th[0]) / 5.0, np.std(np.diff(th))]
    f += [((1 - xs ** 2) * ys ** 2).mean(), (xs ** 2 * ys ** 2).mean(), (ys ** 2).mean(),
          (xs ** 4).mean(), (xs ** 3 * ys).mean()]      # the damping term itself, and friends

    f += [xs[0], ys[0], xs[:5].mean(), ys[:5].mean()]   # estimated initial conditions
    f += sindy(to, xo, yo)
    return f


def feature_names():
    n = ['obs_frac']
    for p in ['xg', 'yg', 'xs', 'ys']:
        n += [f'{p}_{s}' for s in ['std', 'absmean', 'p90', 'absmax', 'kurt', 'skew', 'meansq']]
    n += ['var_x_denoised', 'var_y_denoised', 'corr_xg_yg', 'corr_xs_ys', 'mean_xy']
    for p in ['xs', 'ys', 'energy']:
        n += [f'{p}_blk{i}_std' for i in range(NB)] + [f'{p}_blk{i}_absmean' for i in range(NB)]
        n += [f'{p}_blkstd_delta', f'{p}_blkstd_slope']
    n += [f'env_blk{i}' for i in range(NB)] + ['env_ratio', 'env_delta']
    for p in ['xs', 'xs2', 'ys']:
        n += [f'{p}_nzc', f'{p}_first_zc_t', f'{p}_last_zc_t', f'{p}_period_est']
    n += ['t_argmax_xs', 't_argmin_xs', 'xs_max', 'xs_min', 't_argmax_absys', 'absys_max']
    for p in ['x', 'y']:
        n += [f'LS_{p}_{s}' for s in ['peakfreq', 'peakpow', 'centroid', 'spread', 'entropy']]
        n += [f'LS_{p}_P{i}' for i in range(len(FREQS[::3]))]
    for p in ['xs', 'ys']:
        n += [f'ac_{p}_lag{l}' for l in AC_LAGS] + [f'ac_{p}_first_neg_t', f'ac_{p}_argmin_t']
    n += ['r_mean', 'r_std', 'r_max', 'r_late_minus_early', 'ang_velocity', 'ang_vel_std',
          'mean_(1-x2)y2', 'mean_x2y2', 'mean_y2', 'mean_x4', 'mean_x3y',
          'x0_est', 'y0_est', 'x0_head', 'y0_head', 'sindy_mu', 'sindy_beta', 'sindy_resid']
    return np.array(n)


NAMES = feature_names()
probe = one(A['train']['t'][0], A['train']['x'][0], A['train']['y'][0],
            A['train']['Xg'][0], A['train']['Yg'][0])
assert len(NAMES) == len(probe) == 218, (len(NAMES), len(probe))


def build(split):
    d = A[split]; t0 = time.time()
    F = np.array([one(d['t'][i], d['x'][i], d['y'][i], d['Xg'][i], d['Yg'][i])
                  for i in range(len(d['t']))], dtype=np.float32)
    F = np.nan_to_num(F, nan=0.0, posinf=0.0, neginf=0.0)
    print(f'{split:5s} {F.shape}  in {time.time() - t0:.0f}s '
          f'({1000 * (time.time() - t0) / len(F):.1f} ms/trajectory)', flush=True)
    return F


Ftr, Fte = build('train'), build('test')
print(f'non-finite: {np.count_nonzero(~np.isfinite(Ftr))} / {np.count_nonzero(~np.isfinite(Fte))}')

---
## Block F — models

Three predictors, deliberately different in kind, on one fixed `KFold(10, shuffle=True,
random_state=0)` split so the out-of-fold vectors line up row for row and the blend is honest.

| model | what it uses | why it is here |
|---|---|---|
| median | nothing | the floor: 0.6277 |
| **posterior median** | the recovered generator + the measured `sigma` | no labels at all; the MAE-optimal estimator if the physics is exact |
| **LightGBM, `objective='l1'`** | 218 features + the 34 posterior features | trains directly on the metric; sees the posterior's *shape*, not just its location |
| **dilated 1-D CNN** | raw waveform + posterior scalars, predicting the **residual** | picks up what the posterior misses, and only has to learn a correction |

In [ ]:
med_const = float(np.median(mu))
record('median', np.full(len(mu), med_const), np.full(len(ids_test), med_const), note='the floor')

### F1 — LightGBM

The posterior contributes 34 columns, not one. `po_med` will dominate the gain ranking, but the
width, the skew, the chi-square of the best fit and the local density shape are what let the trees
learn *when to distrust it* — a trajectory whose posterior is broad or whose physics fit is poor
should be pulled back toward the prior, and the trees can only do that if they can see it.

In [ ]:
Gtr = np.hstack([Ftr, Ptr]).astype(np.float32)
Gte = np.hstack([Fte, Pte]).astype(np.float32)
GNAMES = np.concatenate([NAMES, PO_NAMES])
print(f'LightGBM matrix: {Gtr.shape} ({len(NAMES)} engineered + {len(PO_NAMES)} posterior)')

params = dict(objective='l1', metric='l1', learning_rate=0.03, num_leaves=63,
              min_data_in_leaf=40, feature_fraction=0.7, bagging_fraction=0.8,
              bagging_freq=1, lambda_l2=1.0, verbose=-1, num_threads=os.cpu_count())

oof_gbm = np.zeros(len(mu)); test_gbm = np.zeros(len(Gte)); gain = np.zeros(Gtr.shape[1])
t0 = time.time()
for k, (tr, va) in enumerate(folds):
    m = lgb.train(params, lgb.Dataset(Gtr[tr], mu[tr]), CFG['gbm_rounds'],
                  valid_sets=[lgb.Dataset(Gtr[va], mu[va])],
                  callbacks=[lgb.early_stopping(150, verbose=False)])
    oof_gbm[va] = m.predict(Gtr[va], num_iteration=m.best_iteration)
    test_gbm += m.predict(Gte, num_iteration=m.best_iteration) / NFOLD
    gain += m.feature_importance('gain') / NFOLD
    print(f'  fold {k}  MAE {np.abs(oof_gbm[va] - mu[va]).mean():.4f}  '
          f'{m.best_iteration} trees  ({time.time() - t0:.0f}s)', flush=True)

record('lightgbm', oof_gbm, test_gbm, note=f'{Gtr.shape[1]} features')
ckpt('gbm', oof=oof_gbm, test=test_gbm, gain=gain, names=GNAMES)
write_submission(test_gbm, SUBMISSION)
print('  ^ submission.csv upgraded to the GBM; safe to stop here if the session is running out')
rank = np.argsort(-gain)
print('\ntop 15 by gain:')
print(pd.DataFrame({'feature': GNAMES[rank[:15]], 'gain %': 100 * gain[rank[:15]] / gain.sum()})
        .to_string(index=False, float_format=lambda z: f'{z:.1f}'))
print(f"\nposterior block takes {100 * gain[len(NAMES):].sum() / gain.sum():.1f}% of total gain")

### F2 — dilated 1-D CNN, on the residual

Same backbone as before: 8 channels, six residual blocks at dilations 1-2-4-8-1-2 so the receptive
field covers the whole 5 s window, mean + max + attention pooling, L1 loss.

Two changes, both following from §C:

* The posterior scalars are concatenated into the head, so the network does not have to
  rediscover the physics from the waveform.
* The **target is `mu - posterior_median`**, and the prediction is `posterior_median + output`.
  Learning a correction to an estimator that is already close is a far easier problem than
  learning `mu` from scratch, and it leaves the network free to spend its capacity on the
  trajectories where the physics fit is ambiguous.

In [ ]:
def channels(d):
    """(N, 8, 100): raw x/y, two smoothing scales of each, observation density, time ramp."""
    Xg, Yg, Mg = d['Xg'], d['Yg'], d['Mg']
    return np.stack([Xg, Yg,
                     savgol_filter(Xg, 15, 3, axis=1), savgol_filter(Yg, 15, 3, axis=1),
                     savgol_filter(Xg, 31, 3, axis=1), savgol_filter(Yg, 31, 3, axis=1),
                     Mg, np.broadcast_to(np.linspace(0, 1, 100), Xg.shape)], 1).astype(np.float32)


class Feeder:
    """Keeps tensors on the accelerator when they fit, otherwise streams batches to it."""
    def __init__(self, X, S, y=None, limit=6e9):
        self.resident = (X.nbytes + S.nbytes) < limit and DEV != 'cpu'
        self.X = torch.from_numpy(X); self.S = torch.from_numpy(S)
        self.y = None if y is None else torch.from_numpy(np.asarray(y, dtype=np.float32))
        if self.resident:
            self.X = self.X.to(DEV); self.S = self.S.to(DEV)
            if self.y is not None:
                self.y = self.y.to(DEV)
        self.n = len(X)

    def get(self, idx):
        idx = idx.to(self.X.device)
        xb, sb = self.X[idx], self.S[idx]
        yb = None if self.y is None else self.y[idx]
        if not self.resident:
            xb, sb = xb.to(DEV), sb.to(DEV)
            yb = None if yb is None else yb.to(DEV)
        return xb, sb, yb


class Blk(nn.Module):
    def __init__(s, c, d):
        super().__init__()
        s.c1 = nn.Conv1d(c, c, 5, padding=2 * d, dilation=d)
        s.c2 = nn.Conv1d(c, c, 5, padding=2 * d, dilation=d)
        s.n1, s.n2 = nn.BatchNorm1d(c), nn.BatchNorm1d(c)
        s.a = nn.GELU()

    def forward(s, x):
        h = s.a(s.n1(s.c1(x)))
        return s.a(x + s.n2(s.c2(h)))


class Net(nn.Module):
    def __init__(s, cin=8, c=96, ns=0):
        super().__init__()
        s.stem = nn.Sequential(nn.Conv1d(cin, c, 7, padding=3), nn.BatchNorm1d(c), nn.GELU())
        s.blocks = nn.Sequential(*[Blk(c, d) for d in (1, 2, 4, 8, 1, 2)])
        s.att = nn.Conv1d(c, 1, 1)
        s.head = nn.Sequential(nn.Linear(3 * c + ns, 192), nn.GELU(), nn.Dropout(0.1),
                               nn.Linear(192, 1))

    def forward(s, x, sc):
        h = s.blocks(s.stem(x))
        w = torch.softmax(s.att(h), -1)
        z = torch.cat([h.mean(-1), h.amax(-1), (h * w).sum(-1), sc], 1)
        return s.head(z).squeeze(-1)


def predict(net, X, S, bs=1024):
    net.eval(); f = Feeder(X, S); out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            xb, sb, _ = f.get(torch.arange(i, min(i + bs, len(X))))
            out.append(net(xb, sb).float().cpu().numpy())
    return np.concatenate(out)


def train_net(net, X, S, y, epochs, lr, tag='', every=10, va=None):
    """One OneCycle run. Predictions are always taken from the FINAL weights, never the best
    epoch measured on the fold being scored -- that would flatter the number."""
    f = Feeder(X, S, y)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=CFG['weight_decay'])
    bs = CFG['batch']; steps = (f.n + bs - 1) // bs
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, lr, epochs * steps, pct_start=0.15)
    lossf = nn.L1Loss()
    for ep in range(epochs):
        net.train()
        perm = torch.randperm(f.n)
        for i in range(0, f.n, bs):
            xb, sb, yb = f.get(perm[i:i + bs])
            opt.zero_grad()
            loss = lossf(net(xb, sb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 2.0)
            opt.step(); sch.step()
        if va is not None and (ep % every == every - 1 or ep == epochs - 1):
            p = va[2] + predict(net, va[0], va[1])
            print(f'    {tag} ep{ep + 1:3d}  vaMAE {np.abs(p - va[3]).mean():.4f}', flush=True)
    return net

In [ ]:
CHtr, CHte = channels(A['train']), channels(A['test'])
smean, sstd = Ptr.mean(0), Ptr.std(0) + 1e-6          # label-free, so fitting on all of train is fine
Str = ((Ptr - smean) / sstd).astype(np.float32)
Ste = ((Pte - smean) / sstd).astype(np.float32)
base_tr, base_te = PO['train']['med'], PO['test']['med']
resid = (mu - base_tr).astype(np.float32)             # what the CNN actually learns
print(f'CNN inputs: {CHtr.shape} + {Str.shape[1]} posterior scalars on {DEV}')
print(f'residual target: mean {resid.mean():+.4f}  sd {resid.std():.4f}  '
      f'(predicting mu directly would need sd {mu.std():.4f})')

oof_cnn = np.zeros(len(mu)); test_cnn = np.zeros(len(CHte)); seen = np.zeros(len(mu), bool)
t0 = time.time()
for k, (tr, va) in enumerate(folds[:CFG['cnn_folds']]):
    pv = np.zeros(len(va)); pt = np.zeros(len(CHte))
    for s in range(CFG['cnn_seeds']):
        torch.manual_seed(1000 * s + k)
        net = Net(CHtr.shape[1], CFG['cnn_width'], Str.shape[1]).to(DEV)
        train_net(net, CHtr[tr], Str[tr], resid[tr], CFG['cnn_epochs'], CFG['lr'],
                  tag=f'f{k}s{s}', va=(CHtr[va], Str[va], base_tr[va], mu[va]))
        pv += (base_tr[va] + predict(net, CHtr[va], Str[va])) / CFG['cnn_seeds']
        pt += (base_te + predict(net, CHte, Ste)) / CFG['cnn_seeds']
    oof_cnn[va] = pv; seen[va] = True
    test_cnn += pt / CFG['cnn_folds']
    print(f'  fold {k}  MAE {np.abs(pv - mu[va]).mean():.4f}  ({time.time() - t0:.0f}s)', flush=True)

record('cnn (residual)', oof_cnn, test_cnn, mask=seen,
       note=f"{CFG['cnn_epochs']} ep x {CFG['cnn_folds']} folds x {CFG['cnn_seeds']} seeds, final weights")
ckpt('cnn', oof=oof_cnn, test=test_cnn, mask=seen)

---
## Block G — scoreboard, blend, submission

In [ ]:
rows = []
for name, r in RESULTS.items():
    per_fold = [np.abs(r['oof'][va] - mu[va]).mean() for _, va in folds if r['mask'][va].all()]
    rows.append(dict(model=name, MAE=r['mae'], RMSE=r['rmse'], **{'RMSE/MAE': r['rmse'] / r['mae']},
                     folds=len(per_fold),
                     fold_min=min(per_fold) if per_fold else np.nan,
                     fold_max=max(per_fold) if per_fold else np.nan,
                     **{'x floor': r['mae'] / BAYES_FLOOR}, note=r['note']))
board = pd.DataFrame(rows).sort_values('MAE').reset_index(drop=True)
display(board.style.hide(axis='index').format({
    'MAE': '{:.4f}', 'RMSE': '{:.4f}', 'RMSE/MAE': '{:.2f}',
    'fold_min': '{:.4f}', 'fold_max': '{:.4f}', 'x floor': '{:.2f}'}))
print(f'Bayes floor (simulated, sigma = {SIG_MODEL:.3f}): {BAYES_FLOOR:.4f}')
print('\nCaveats worth keeping in view:')
print('  * LightGBM early-stops on the fold it is scored on, so its MAE is mildly optimistic.')
print('  * CNN numbers use final-epoch weights, so they are NOT flattered by best-epoch selection.')
print('  * The Bayes floor assumes the recovered generator is exact; it is a target, not a law.')
print('  * Every number here is measured at the TEST noise level, so CV is comparable to the board.')

In [ ]:
trained = [n for n in RESULTS if n != 'median' and RESULTS[n]['mask'].sum() > 0]
palette = dict(zip(trained, [BLUE, ORANGE, GREEN, PURPLE][:len(trained)]))

fig, axes = plt.subplots(1, 3, figsize=(12.6, 3.9), gridspec_kw=dict(width_ratios=[1.15, 1, 1]))
ax = axes[0]
for n in trained:
    r = RESULTS[n]
    fm = [np.abs(r['oof'][va] - mu[va]).mean() if r['mask'][va].all() else np.nan for _, va in folds]
    ax.plot(range(NFOLD), fm, color=palette[n], marker='o', ms=4, mfc=SURFACE, mew=1.4, label=n)
ax.axhline(BAYES_FLOOR, color=MUTED, lw=1.2, ls='--')
ax.text(0.02, BAYES_FLOOR, ' Bayes floor', color=MUTED, va='bottom', transform=ax.get_yaxis_transform())
ax.set(xlabel='fold', ylabel='MAE', title='per-fold MAE: is the ranking stable?')
ax.legend(loc='best')

ax = axes[1]
ed = np.linspace(0.5, 3.0, 11); cen = 0.5 * (ed[1:] + ed[:-1])
for n in trained:
    r = RESULTS[n]; m = r['mask']
    err = [np.abs(r['oof'][m & (mu >= a) & (mu < b)] - mu[m & (mu >= a) & (mu < b)]).mean()
           for a, b in zip(ed[:-1], ed[1:])]
    ax.plot(cen, err, color=palette[n], lw=2, label=n)
ax.set(xlabel='true mu', ylabel='mean |error|', title='where each model struggles')
ax.legend(loc='best')

ax = axes[2]
best = board[board.model != 'median'].model.iloc[0]
other = [n for n in trained if n != best]
if other:
    o = other[0]
    cm = RESULTS[best]['mask'] & RESULTS[o]['mask']
    ra, rb = RESULTS[best]['oof'][cm] - mu[cm], RESULTS[o]['oof'][cm] - mu[cm]
    hb = ax.hexbin(ra, rb, gridsize=45, cmap=CMAP, mincnt=1, linewidths=0)
    ax.set(xlabel=f'residual, {best}', ylabel=f'residual, {o}',
           title=f'residual correlation {np.corrcoef(ra, rb)[0, 1]:.2f}')
    cb = fig.colorbar(hb, ax=ax, pad=0.02); cb.outline.set_visible(False)
    cb.ax.tick_params(color=MUTED, labelcolor=MUTED)
plt.tight_layout(); plt.show()

print('residual correlation between models (lower = more to gain from blending):')
Cm = pd.DataFrame({a: {b: np.corrcoef(RESULTS[a]['oof'][RESULTS[a]['mask'] & RESULTS[b]['mask']]
                                      - mu[RESULTS[a]['mask'] & RESULTS[b]['mask']],
                                      RESULTS[b]['oof'][RESULTS[a]['mask'] & RESULTS[b]['mask']]
                                      - mu[RESULTS[a]['mask'] & RESULTS[b]['mask']])[0, 1]
                       for b in trained} for a in trained})
print(Cm.to_string(float_format=lambda z: f'{z:.2f}'))

In [ ]:
usable = sorted([n for n in RESULTS if n != 'median' and RESULTS[n]['test'] is not None],
                key=lambda n: RESULTS[n]['mae'])
common = np.ones(len(mu), bool)
for n in usable:
    common &= RESULTS[n]['mask']
print(f'blending {usable} on {common.sum()} commonly covered rows')

P = np.column_stack([RESULTS[n]['oof'][common] for n in usable])
T = np.column_stack([RESULTS[n]['test'] for n in usable])
target = mu[common]


def simplex(k, step=0.05):
    """All non-negative weight vectors on a grid that sum to 1."""
    if k == 1:
        yield (1.0,); return
    n = int(round(1 / step))
    for head in range(n + 1):
        for rest in simplex(k - 1, step):
            yield (head * step,) + tuple(r * (1 - head * step) for r in rest)


cands = np.array(list(simplex(len(usable))))
maes = np.concatenate([np.abs(P @ cands[i:i + 400].T - target[:, None]).mean(0)
                       for i in range(0, len(cands), 400)])
W = cands[maes.argmin()]
blend_oof = np.clip(P @ W, 0.5, 3.0)
blend_test = np.clip(T @ W, 0.5, 3.0)

print('\nweights: ' + ', '.join(f'{n} {w:.2f}' for n, w in zip(usable, W)))
singles = {n: np.abs(RESULTS[n]['oof'][common] - target).mean() for n in usable}
bn = min(singles, key=singles.get)
mae_b = float(np.abs(blend_oof - target).mean())
print(f'blend  MAE {mae_b:.4f}   RMSE {np.sqrt(((blend_oof - target) ** 2).mean()):.4f}')
print(f'best single on the same rows: {bn} {singles[bn]:.4f}  ->  blend gains '
      f'{100 * (1 - mae_b / singles[bn]):.1f}%')
print(f'clipping moved {np.mean((P @ W < 0.5) | (P @ W > 3.0)):.2%} of predictions')
print(f'\nvs the Bayes floor {BAYES_FLOOR:.4f}: blend is {mae_b / BAYES_FLOOR:.2f}x it '
      f'({mae_b - BAYES_FLOOR:+.4f})')
print(f'vs the median floor 0.6277: {0.6277 / mae_b:.1f}x better')

In [ ]:
sub = write_submission(blend_test, SUBMISSION)
assert len(sub) == len(pd.read_csv(f'{DATA}/sample_submission.csv')), (
    f'submission has {len(sub)} rows but sample_submission has '
    f'{len(pd.read_csv(f"{DATA}/sample_submission.csv"))} -- this is a subsampled run, do NOT submit it')
print(f'final submission: {os.path.abspath(SUBMISSION)}')
print(sub.head().to_string(index=False))
print(f'\npredicted test mu: mean {sub.target.mean():.3f}  std {sub.target.std():.3f}  '
      f'range [{sub.target.min():.3f}, {sub.target.max():.3f}]')
print(f'train mu for comparison: mean {mu.mean():.3f}  std {mu.std():.3f}')

---
## What to read off this run

1. **The blend against the Bayes floor.** That ratio is the only number that says whether more
   modelling is worth anything. Close to 1.0 means the recovered physics has been fully exploited
   and the residual error is irreducible measurement noise; well above it means the estimator, not
   the information, is the limit.
2. **How much gain the posterior block takes in LightGBM.** If `po_med` dominates and the rest of
   the posterior block contributes little, the trees are not finding the "when to distrust it"
   signal, and the width features are dead weight.
3. **The residual correlation.** The CNN now starts from the posterior, so it is expected to be
   more correlated with the other two than the old waveform-only CNN was (0.78). If it exceeds
   ~0.95 the blend has stopped buying anything and the seeds would be better spent elsewhere.
4. **`po_chi2`.** Its median should sit near 1.0 — the fit reaching the noise floor. Materially
   above 1.0 means the generative model is still misspecified, and `beta` is the first suspect.

### If there is budget for a second run

In rough order of expected value:

* **Let `beta` vary per trajectory.** §C0 fits one global value. If the generator actually drew
  `beta` per trajectory, a global fit is a compromise and the per-trajectory posterior is
  mis-specified in a way no amount of training data fixes. Testing it costs one extra sweep:
  refit `beta` within `mu` bins and see whether the optimum moves.
* **Two noise realisations of train** for the CNN, instead of one. The lift in §B2 is a random
  draw; averaging over two doubles the CNN cost but removes that draw from the result.
* **Refine the `mu` grid adaptively** near the posterior peak rather than uniformly — the median is
  only as accurate as the grid around the crossing point, and most of the 321 grid points are
  spent far out in the tails where the density is negligible.